In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "Google API key not found. Check your .env file."
    )

print("Google API key loaded successfully.")

In [61]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Indexing

In [62]:
import re

DEFAULT_VIDEO_ID = "rHLEWRxRGiM"
DEFAULT_LANGUAGE = "en"
MAX_TRANSCRIPT_CHARS = 50_000 # limitation of characters

def extract_video_id(url_or_id: str) -> str:
    url_or_id = url_or_id.strip()
    match = re.search(
        r"(?:v=|youtu\.be/|youtube\.com/shorts/|youtube\.com/embed/)([A-Za-z0-9_-]{11})",
        url_or_id
    )
    if match:
        return match.group(1)
    if len(url_or_id) == 11 and re.match(r"^[A-Za-z0-9_-]{11}$", url_or_id):
        return url_or_id
    return None



In [63]:
api = YouTubeTranscriptApi()
url_input = input("Enter YouTube video URL: ").strip()
video_id = extract_video_id(url_input)


Enter YouTube video URL:  LPZh9BOjkQs


In [64]:
# Validate video and check available transcripts
transcripts_list = None
use_default = False

if video_id:
    try:
        transcripts_list = list(api.list(video_id))
    except Exception as e:
        print(f"Could not load transcripts for video ({video_id}): {e}")

# going to default if invalid or no transcripts available
if not transcripts_list:
    
    print("\n[!] Invalid URL or no accessible transcripts found.")
    print(f"Switching to default video ID: {DEFAULT_VIDEO_ID} in English.")
    video_id = DEFAULT_VIDEO_ID
    use_default = True
    transcripts_list = list(api.list(video_id))


In [65]:
transcripts_list

In [66]:
#select transcript language (or auto-select for default)

chosen_lang_code = DEFAULT_LANGUAGE

if use_default:
    chosen_lang_code = DEFAULT_LANGUAGE
else:
    print(f"\nAvailable Transcripts for Video ID: {video_id}")
    for idx, t in enumerate(transcripts_list):
        tag = "[Auto-Generated]" if t.is_generated else "[Manual]"
        print(f"  [{idx + 1}] {t.language} ({t.language_code}) {tag}")

    while True:
        choice = input(f"\nSelect a transcript number (1-{len(transcripts_list)}) or enter language code: ").strip()
        if choice.isdigit() and 1 <= int(choice) <= len(transcripts_list):
            chosen_lang_code = transcripts_list[int(choice) - 1].language_code
            break
        elif any(t.language_code.lower() == choice.lower() for t in transcripts_list):
            chosen_lang_code = choice.lower()
            break
        else:
            print("Invalid selection. Please try again.")

print(f"\nFetching transcript in: '{chosen_lang_code}'...")


Available Transcripts for Video ID: LPZh9BOjkQs
  [1] Arabic (ar) [Manual]
  [2] Chinese (zh) [Manual]
  [3] English (en) [Manual]
  [4] Finnish (fi) [Manual]
  [5] French (fr) [Manual]
  [6] Hungarian (hu) [Manual]
  [7] Indonesian (id) [Manual]
  [8] Italian (it) [Manual]
  [9] Japanese (ja) [Manual]
  [10] Korean (ko) [Manual]
  [11] Polish (pl) [Manual]
  [12] Portuguese (Brazil) (pt-BR) [Manual]
  [13] Russian (ru) [Manual]
  [14] Spanish (es) [Manual]
  [15] Thai (th) [Manual]
  [16] Turkish (tr) [Manual]
  [17] Ukrainian (uk) [Manual]
  [18] English (auto-generated) (en) [Auto-Generated]



Select a transcript number (1-18) or enter language code:  en



Fetching transcript in: 'en'...


In [67]:
raw_snippets = api.fetch(video_id, languages=[chosen_lang_code])

In [68]:
raw_snippets

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[Submit subtitle corrections at criblate.com]\nImagine you happen across a short movie script that', start=1.14, duration=2.836), FetchedTranscriptSnippet(text='describes a scene between a person and their AI assistant.', start=3.976, duration=3.164), FetchedTranscriptSnippet(text="The script has what the person asks the AI, but the AI's response has been torn off.", start=7.48, duration=5.58), FetchedTranscriptSnippet(text='Suppose you also have this powerful magical machine that can take', start=13.06, duration=3.92), FetchedTranscriptSnippet(text='any text and provide a sensible prediction of what word comes next.', start=16.98, duration=3.98), FetchedTranscriptSnippet(text='You could then finish the script by feeding in what you have to the machine,', start=21.5, duration=4.006), FetchedTranscriptSnippet(text="seeing what it would predict to start the AI's answer,", start=25.506, duration=2.862), FetchedTranscriptSnippet(te

In [69]:
len(raw_snippets)

117

In [70]:
filtered_snippets = []

for s in raw_snippets:
    if isinstance(s, dict):
        text = s.get("text", "")
    else:
        text = getattr(s, "text", "")

    if text:
        filtered_snippets.append(text)

In [71]:
full_transcript_text = " ".join(filtered_snippets)

# Limit the transcript to 500,000 characters
if len(full_transcript_text) > MAX_TRANSCRIPT_CHARS:
    full_transcript_text = full_transcript_text[:MAX_TRANSCRIPT_CHARS]
    print("Warning: Transcript truncated to 500,000 characters.")

if not full_transcript_text.strip():
    raise ValueError("The transcript is empty.")

print(
    f"Transcript loaded successfully: "
    f"{len(full_transcript_text):,} characters."
)

Transcript loaded successfully: 7,585 characters.


In [72]:
full_transcript_text

"[Submit subtitle corrections at criblate.com]\nImagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interaction between a us

In [73]:
# Text Splitting

splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks = splitter.create_documents(
    texts=[full_transcript_text],
    metadatas=[{"video_id": video_id, "language": chosen_lang_code}]
)

In [74]:
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 11


In [75]:
# Embedding and storing in VectorStore
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    output_dimensionality=768
)
vector_store = FAISS.from_documents(chunks, embeddings)

In [76]:
vector_store.index_to_docstore_id

{0: 'bb970355-eb1e-4e6e-9f72-5f6ebc4dcd73',
 1: 'f392fe75-9d77-4246-845c-7168c668f525',
 2: '034297cb-24a9-4bf1-8625-b8c7c52a38f1',
 3: 'ae9e4c97-60d6-45b6-99b3-ae6a179e6f3a',
 4: '43ae13cd-375e-48ea-91eb-896ffe9df9b9',
 5: '940c98d6-3909-4384-a596-4ff10d48655d',
 6: '1f268811-5180-41b7-beaf-32cf7e8d844e',
 7: '18f1933d-12e3-4755-8d47-3e20c09b7037',
 8: '2226d3f7-5c7c-4a28-bc20-25db3028f192',
 9: 'f2b15835-e0a4-4f6c-8d44-deeec7296a34',
 10: '2bd2e012-2c41-464e-8836-67baf081d938'}

In [77]:
vector_store.get_by_ids(['2226d3f7-5c7c-4a28-bc20-25db3028f192'])

[Document(id='2226d3f7-5c7c-4a28-bc20-25db3028f192', metadata={'video_id': 'LPZh9BOjkQs', 'language': 'en'}, page_content="typically also include a second type of operation known as a feed-forward neural network, and this gives the model extra capacity to store more patterns about language learned during training. All of this data repeatedly flows through many different iterations of these two fundamental operations, and as it does so, the hope is that each list of numbers is enriched to encode whatever information might be needed to make an accurate prediction of what word follows in the passage. At the end, one final function is performed on the last vector in this sequence, which now has had a chance to be influenced by all the other context from the input text, as well as everything the model learned during training, to produce a prediction of the next word. Again, the model's prediction looks like a probability for every possible next word. Although researchers design the framewor

## Retrieval

In [78]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [79]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000016DBF4CFE50>, search_kwargs={'k': 3})

## Augmentation

In [80]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [81]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.2
)

In [82]:
# Prompt for answering questions based on context, language, and conversation summary

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful and accurate assistant answering questions about a YouTube video.
Rules:
1. Answer ONLY using information directly stated in the provided Transcript Context.
2. If the answer cannot be determined from the context, state clearly: "The provided video transcript does not contain this information."
3. Respond strictly in the requested language: {answer_language}.
4. Refer to the Conversation Summary below if the user refers to earlier parts of the conversation.
---
Transcript Context:
{context}
---
Conversation Summary so far:
{chat_history}
"""),
    ("human", "{question}")
])

# Prompt for progressively updating the conversation summary
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant maintaining a concise conversation summary (2-3 sentences max). Combine the current summary with the latest Q&A exchange."),
    ("human", """Current Summary: {chat_history}
Latest Exchange:
User: {question}
AI: {answer}
Updated Concise Summary:""")
])

In [83]:
parser = StrOutputParser()

summary_chain = summary_prompt | llm | parser

In [84]:
# used to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [85]:
chain = (
    RunnableParallel({
        "context": itemgetter("question") | retriever | RunnableLambda(format_docs),
        "question": itemgetter("question"),
        "answer_language": RunnableLambda(lambda x: x.get("answer_language") or "English"),
        "chat_history": RunnableLambda(lambda x: x.get("chat_history") or "No previous conversation.")
    })
    | prompt
    | llm
    | parser
)

In [86]:
print("=" * 60)
print(f"Chat session started for Video ID: {video_id}")
print("Type 'exit' or 'quit' to end the session.\n")

Chat session started for Video ID: LPZh9BOjkQs
Type 'exit' or 'quit' to end the session.



In [87]:
conversation_summary = "No previous conversation."

In [88]:
while True:
    user_q = input("\nEnter your question: ").strip()
    if not user_q:
        print("Please enter a non-empty question.")
        continue
    if user_q.lower() in ["exit", "quit", "q"]:
        print("Exiting chat. Goodbye!")
        break

    target_lang = input("Preferred answer language (e.g. English, Hindi, Telugu) [Press Enter for English]: ").strip()
    if not target_lang:
        target_lang = "English"

    # # Invoke the chain
    answer = chain.invoke({
        "question": user_q,
        "answer_language": target_lang,
        "chat_history": conversation_summary
    })

    print(f"\n[Answer in {target_lang}]:\n{answer}")

    # Update the conversation summary for context continuity
    conversation_summary = summary_chain.invoke({
        "chat_history": conversation_summary,
        "question": user_q,
        "answer": answer
    })

    


Enter your question:  what does attention means?
Preferred answer language (e.g. English, Hindi, Telugu) [Press Enter for English]:  Telugu


C:\Users\THARUN TEJA S\.conda\envs\ai_research_agent\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Answer in Telugu]:
ట్రాన్స్‌ఫార్మర్లలో మరియు ఇతర లాంగ్వేజ్ మోడళ్లలో ఉపయోగించే "అటెన్షన్" (attention) అనేది ఒక ప్రత్యేకమైన ఆపరేషన్. ఈ ఆపరేషన్ ద్వారా పదాలను సూచించే సంఖ్యల జాబితాలు అన్నీ ఒకదానితో ఒకటి మాట్లాడుకునే అవకాశం లభిస్తుంది. దీనివల్ల చుట్టూ ఉన్న సందర్భాన్ని (context) బట్టి ఆ పదాలు సూచించే అర్ధాలను మరింత మెరుగుపరుచుకుంటాయి, ఇవన్నీ సమాంతరంగా (in parallel) జరుగుతాయి. ఉదాహరణకు, "bank" అనే పదానికి సంబంధించిన సంఖ్యలు, దాని చుట్టూ ఉన్న సందర్భాన్ని బట్టి మరింత ఖచ్చితమైన "riverbank" (నది ఒడ్డు) అనే భావనను సూచించేలా మారుతాయి.


C:\Users\THARUN TEJA S\.conda\envs\ai_research_agent\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Enter your question:  what does pretraining means simply in 5 lines
Preferred answer language (e.g. English, Hindi, Telugu) [Press Enter for English]:  


...

C:\Users\THARUN TEJA S\.conda\envs\ai_research_agent\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Answer in English]:
Based on the provided transcript, transformers handle word ambiguity through a special operation known as **attention**. This operation allows the lists of numbers representing words to "talk to one another and refine the meanings they encode based on the context around, all done in parallel." 

For example, the numbers encoding the word **"bank"** might be changed based on the surrounding context to specifically encode the notion of a **"riverbank."**

The provided video transcript does not contain further specific examples related to this.


C:\Users\THARUN TEJA S\.conda\envs\ai_research_agent\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Enter your question:  exit


Exiting chat. Goodbye!
